# Mean-difference probe — fast single-session submission

An **efficiency-optimized** version of `baselines/mean_probe.ipynb`. Same probe,
same activations, but:

1. **One NDIF session** for the whole dataset (`util.run_full_session`) — one
   queue wait instead of one remote trace per batch.
2. **Early-stop at the probe's layer** (`tracer.stop()`): the probe reads layer
   26, so we skip every later transformer layer **and the LM head** — roughly
   half the forward pass of a ~62-layer model is never computed.
3. **Correct contract**: writes `index,deceptive,score` keyed on the dataset's
   `index` column (the stock baseline wrote `id,prediction` on row position,
   which the grader rejects).
4. **Faithful probe scoring**: applies the standardization the probe was trained
   with (`mu`/`sigma`/`bias`/`score_std` saved in the `.npz`) — the stock
   baseline ignored those fields and projected raw activations.


In [ ]:
import os

# ── DO NOT CHANGE ───────────────────────────────────────────────────
# The runner sets these for each run; read them, don't hard-code or override.
#   DATASET_NAME : the eval dataset to predict on.
#   NDIF_HOST / NDIF_API_KEY / HF_TOKEN : injected; nnsight + HF read them.
DATASET_NAME = os.environ["DATASET_NAME"]

## Load the trained probe

`mean_probe.npz` ships next to this notebook. It carries the probe **direction**
plus everything needed to reproduce the exact feature extraction and scoring:
`layer`, `model_id`, `max_seq_len`, and the standardization stats
(`mu`, `sigma`, `normalize`, `bias`, `score_std`).


In [ ]:
import numpy as np

CANDIDATES = ["submission/mean_probe.npz", "mean_probe.npz", "baselines/mean_probe.npz"]
WEIGHTS = next((p for p in CANDIDATES if os.path.exists(p)), None)
assert WEIGHTS is not None, f"mean_probe.npz not found; looked in {CANDIDATES}"

z = np.load(WEIGHTS)
direction = z["direction"].astype(np.float32)               # (hidden,)
LAYER     = int(z["layer"])
MODEL_ID  = str(z["model_id"])
MAX_LEN   = int(z["max_seq_len"])
# Standardization metadata (present in the shipped probe; default to a no-op if absent).
NORMALIZE = bool(z["normalize"]) if "normalize" in z.files else False
MU        = z["mu"].astype(np.float32)        if "mu" in z.files        else np.zeros_like(direction)
SIGMA     = z["sigma"].astype(np.float32)     if "sigma" in z.files     else np.ones_like(direction)
BIAS      = float(z["bias"])                  if "bias" in z.files      else 0.0
SCORE_STD = float(z["score_std"])             if "score_std" in z.files else 1.0
print(f"probe: model={MODEL_ID} | layer={LAYER} | dim={direction.shape[0]} | "
      f"max_len={MAX_LEN} | normalize={NORMALIZE} | score_std={SCORE_STD:.3g}")

## Method

`detect` runs **on NDIF**. It opens one trace per batch, reads the residual
stream at `LAYER`, mean-pools over each row's response span, applies the probe
(standardize → project → sigmoid), saves the scores, then `tracer.stop()`s so the
rest of the network is never run.

Per the `util` rules, `detect` is cloudpickled to NDIF, so it only uses
`from`-imported names, `batch` methods, `torch`, `**kwargs`, and `model`. The
probe arrays are passed in through `run_full_session(..., **kwargs)` as tensors.


In [ ]:
import sys
sys.path.insert(0, "submission")
import torch
from util import build_model, decoder_layers, run_full_session, load_examples, write_submission, response_span


def probe_preprocess(messages, tokenizer, max_len):
    """Replicate the probe's training tokenization so activations match:
      * merge a leading system turn into the first user turn,
      * trim the conversation to end at the last assistant message (the judged one),
      * locate the response span and drop the trailing <end_of_turn>,
      * left-trim to max_len, preserving the response.
    Returns (token_ids, (start, end))."""
    msgs = list(messages)
    # trim to last assistant turn
    a = [i for i, m in enumerate(msgs) if m.get("role") == "assistant"]
    if a:
        msgs = msgs[: a[-1] + 1]
    # merge leading system into first user
    norm, pending = [], None
    for m in msgs:
        if m["role"] == "system" and not norm:
            pending = m.get("content", ""); continue
        if not norm and m["role"] == "user" and pending is not None:
            norm.append({"role": "user", "content": pending + "\n\n" + m.get("content", "")})
            pending = None; continue
        norm.append(m)
    if not norm:
        norm = msgs
    ids, s, e = response_span(norm, tokenizer)
    # strip trailing end-of-turn tokens from the response span
    eot = tokenizer.convert_tokens_to_ids("<end_of_turn>")
    while e > s and ids[e - 1] == eot:
        e -= 1
    # left-trim to max_len, keeping the response
    if max_len and len(ids) > max_len:
        cut = min(len(ids) - max_len, s)
        ids, s, e = ids[cut:], s - cut, e - cut
        if len(ids) > max_len:
            cut2 = len(ids) - max_len
            ids, s, e = ids[cut2:], max(0, s - cut2), e - cut2
    if e <= s:                      # empty span -> pool the whole sequence as a fallback
        s, e = 0, len(ids)
    return ids, (s, e)


def detect(model, model_id, lora_id, batch, *, direction, mu, sigma, bias, score_std, normalize, layer):
    """(B,) deception probabilities in [0,1], computed remotely at one layer."""
    with model.trace({"input_ids": batch.input_ids,
                      "attention_mask": batch.attention_mask}) as tracer:
        h = decoder_layers(model)[layer].output            # (B, seq, hidden)
        feats = batch.pool_response(h)                      # (B, hidden), mean over response
        dev = feats.device
        x = feats.float()
        if normalize:
            x = (x - mu.to(dev)) / sigma.to(dev)
        s = (x @ direction.to(dev) + bias) / score_std     # (B,)
        scores = torch.sigmoid(s).save()
        tracer.stop()                                       # skip layers > `layer` + LM head
    return scores

## Run one session and write `submission.csv`

`batch_size * max_len` stays under the ~2560-token budget for the 27B model
(`8 * 300 = 2400`). The probe is a fixed feature extractor trained on
`google/gemma-3-27b-it`, so we force that model (no LoRA) for **every** dataset,
matching how the baseline probe is applied.


In [ ]:
model = build_model(MODEL_ID)          # probe base; fixed feature extractor, no LoRA

# Optional `--limit N` (forwarded as $ALETHEIA_LIMIT): score only the first N rows
# for a fast rehearsal. Unset on the leaderboard -> scores every row.
LIMIT = int(os.environ["ALETHEIA_LIMIT"]) if os.environ.get("ALETHEIA_LIMIT") else None

probe_kwargs = dict(
    direction=torch.tensor(direction),
    mu=torch.tensor(MU),
    sigma=torch.tensor(SIGMA),
    bias=float(BIAS),
    score_std=float(SCORE_STD),
    normalize=NORMALIZE,
    layer=LAYER,
)

scores = run_full_session(
    DATASET_NAME, detect,
    preprocess=probe_preprocess,
    model=model,
    batch_size=8,
    max_len=MAX_LEN,
    remote=True,
    limit=LIMIT,
    **probe_kwargs,
)
print(f"scored {len(scores)} rows in one session")

examples = load_examples(DATASET_NAME)
write_submission(list(examples["index"])[:len(scores)], scores)